# Smart Waste – Export Model ke ONNX

Mengubah `waste_classifier.keras` hasil `train_waste_classifier.ipynb` menjadi `waste_classifier.onnx` supaya bisa dijalankan langsung di backend Elysia (Bun) dengan `onnxruntime`.

Notebook ini juga **membuktikan model ONNX identik** dengan model Keras: keduanya dijalankan pada 900 gambar test lalu hasilnya dibandingkan.

**Cukup CPU**, tidak perlu GPU. Jalankan semua sel (**Runtime → Run all**).

## 1. Setup

In [ ]:
!pip install -q tf2onnx onnx onnxruntime

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/smart-waste"
ZIP_PATH = f"{DRIVE_DIR}/dataset_6class.zip"
DATA_DIR = "/content/data"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"

KERAS_PATH = f"{OUTPUT_DIR}/waste_classifier.keras"
ONNX_PATH = f"{OUTPUT_DIR}/waste_classifier.onnx"
REFERENCE_PATH = f"{OUTPUT_DIR}/onnx_reference.json"

!mkdir -p "{DATA_DIR}"
!unzip -q -o "{ZIP_PATH}" -d "{DATA_DIR}"
!ls -lh "{OUTPUT_DIR}"

In [ ]:
import json
import os

import keras
import numpy as np
import onnxruntime as ort
import tensorflow as tf

IMG_SIZE = 224

with open(f"{OUTPUT_DIR}/labels.json") as f:
    CLASS_NAMES = json.load(f)

model = keras.models.load_model(KERAS_PATH)
print("Kelas:", CLASS_NAMES)
print([layer.name for layer in model.layers])

## 2. Model Khusus Prediksi

Layer augmentasi hanya dipakai saat training, jadi dibuang sebelum export supaya graph ONNX lebih sederhana. Layer lainnya dipakai ulang, sehingga bobotnya sama persis dengan model hasil training.

Input: gambar RGB 224×224, nilai piksel 0–255 (float32), format `[batch, 224, 224, 3]`.
Output: 6 probabilitas sesuai urutan `labels.json`.

In [ ]:
inference_model = keras.Sequential(
    [keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")]
    + [layer for layer in model.layers[1:] if layer.name != "augmentation"],
    name="waste_classifier_inference",
)
inference_model.summary()

## 3. Export ke ONNX

In [ ]:
try:
    # Keras 3 bisa langsung export ke ONNX
    inference_model.export(ONNX_PATH, format="onnx")
    print("Export langsung dari Keras berhasil")
except Exception as e:
    print("Export langsung gagal, pakai jalur SavedModel + tf2onnx:", e)
    SAVED_DIR = "/content/saved_model"
    inference_model.export(SAVED_DIR, format="tf_saved_model")
    !python -m tf2onnx.convert --saved-model "{SAVED_DIR}" --output "{ONNX_PATH}" --opset 17

print(f"{ONNX_PATH}: {os.path.getsize(ONNX_PATH) / 1024 / 1024:.1f} MB")

In [ ]:
session = ort.InferenceSession(ONNX_PATH)
INPUT_NAME = session.get_inputs()[0].name
for i in session.get_inputs():
    print("input :", i.name, i.shape, i.type)
for o in session.get_outputs():
    print("output:", o.name, o.shape, o.type)

## 4. Verifikasi: Keras vs ONNX pada 900 Gambar Test

In [ ]:
test_ds = keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "test"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode="int",
    shuffle=False,
)
assert test_ds.class_names == CLASS_NAMES, "Urutan kelas berbeda dengan labels.json"
test_file_paths = test_ds.file_paths

keras_probs, onnx_probs, y_true = [], [], []
for images, labels in test_ds:
    batch = images.numpy().astype(np.float32)
    keras_probs.append(inference_model.predict(batch, verbose=0))
    onnx_probs.append(session.run(None, {INPUT_NAME: batch})[0])
    y_true.append(labels.numpy())

keras_probs = np.concatenate(keras_probs)
onnx_probs = np.concatenate(onnx_probs)
y_true = np.concatenate(y_true)

keras_pred = keras_probs.argmax(axis=1)
onnx_pred = onnx_probs.argmax(axis=1)

print(f"Akurasi Keras : {(keras_pred == y_true).mean():.4f}")
print(f"Akurasi ONNX  : {(onnx_pred == y_true).mean():.4f}")
print(f"Prediksi sama : {(keras_pred == onnx_pred).sum()} / {len(y_true)}")
print(f"Selisih probabilitas maksimum: {np.abs(keras_probs - onnx_probs).max():.2e}")

assert (keras_pred == onnx_pred).all(), "Ada prediksi yang berbeda antara Keras dan ONNX"
print("\n✅ Model ONNX identik dengan model Keras")

## 5. Data Referensi untuk Uji di Bun

Menyimpan probabilitas ONNX untuk 12 gambar test (2 per kelas). Script `ml/onnx-check` di laptop menjalankan model ONNX yang sama pada gambar yang sama lalu membandingkannya dengan file ini.

In [ ]:
picked = []
for cls_idx in range(len(CLASS_NAMES)):
    picked += list(np.where(y_true == cls_idx)[0][:2])

reference = {
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "samples": [
        {
            "file": os.path.relpath(test_file_paths[i], DATA_DIR).replace(os.sep, "/"),
            "label": CLASS_NAMES[y_true[i]],
            "probs": onnx_probs[i].round(6).tolist(),
        }
        for i in picked
    ],
}
with open(REFERENCE_PATH, "w") as f:
    json.dump(reference, f, indent=2)

print(f"{len(picked)} sampel disimpan ke {REFERENCE_PATH}")
!ls -lh "{OUTPUT_DIR}"